## Databricks SQL AI Functions

Databricks provides **built-in SQL AI functions** powered by foundation models for text analysis, classification, extraction, and generation — no endpoint setup required. These functions run directly in SQL queries and work on any string column or literal, making it easy to add intelligence to your data pipelines.

> Disclaimer: If some of the AI functions give an error, update the cluster DBR version, the current DBR might not support all the functions. For serverless, please update the environment version.

## `ai_gen()` — Generate Text from Prompts

Generates text responses from natural language prompts using a built-in foundation model. Ideal for content creation, email drafting, and on-the-fly text generation.

In [0]:
%sql
SELECT ai_gen('Generate a concise, cheerful email subject line for a summer bike sale with 20% discount') AS generated_text;

## `ai_query()` — Query Foundation Models with Control

Queries a specific foundation model endpoint with full control over model parameters (temperature, max_tokens) and supports structured output via `responseFormat`.

In [0]:
%sql
SELECT ai_query('databricks-meta-llama-3-3-70b-instruct', 'Explain what a lakehouse architecture is in exactly 3 bullet points.') AS response;

## `ai_classify()` — Classify Text into Custom Labels

Classifies text into one or more custom labels. Supports label descriptions and global instructions for high-accuracy categorization. Always use v2 (`MAP('version', '2.0')`).

In [0]:
%sql
SELECT 
  text,
  ai_classify(
    text,
    '{
      "billing_error": "Payment, invoice, or refund issues", 
      "product_defect": "Any malfunction, bug, or breakage", 
      "account_issue": "Login failures, password resets", 
      "feature_request": "Customer suggestions for improvements"}',
    MAP('version', '2.0', 'instructions', 'Classify customer support tickets by primary issue.')
  ):response[0]::STRING AS classification
FROM (VALUES 
  ('I cannot log into my account after resetting my password'),
  ('The screen on my new laptop cracked after one day'),
  ('I was charged twice for my subscription this month'),
  ('It would be great if you added dark mode to the app')
) AS t(text);

## `ai_extract()` — Extract Structured Data from Text

Extracts typed fields from unstructured text using a JSON schema. Supports nested objects, arrays, enums, and field descriptions for precise extraction.

In [0]:
%sql
with extracted as (
SELECT ai_extract(
  'Invoice #INV-2024-0892 from Contoso Ltd for 4,350.00 dollars dated 2024-03-15. Payment terms: Net 30. Contact: john.smith@contoso.com',
  '{
    "invoice_id": {"type": "string", "description": "Unique invoice identifier"},
    "vendor_name": {"type": "string", "description": "Legal business name"},
    "total_amount": {"type": "number", "description": "Total invoice amount in dollars"},
    "invoice_date": {"type": "string", "description": "Date in YYYY-MM-DD format"},
    "payment_terms": {"type": "string"},
    "contact_email": {"type": "string"}
  }'
) AS extracted_data
)
SELECT
  extracted_data:response:contact_email:value::STRING as contact_email,
  extracted_data:response:invoice_date:value::DATE as invoice_date,
  extracted_data:response:invoice_id:value::STRING as invoice_id,
  extracted_data:response:payment_terms:value::STRING as payment_terms,
  extracted_data:response:total_amount:value::DOUBLE as total_amount,
  extracted_data:response:vendor_name:value::STRING as vendor_name
FROM extracted

## `ai_analyze_sentiment()` — Detect Sentiment

Analyzes text and returns sentiment as positive, negative, neutral, or mixed. Works on individual strings or across table columns.

In [0]:
%sql
SELECT 
  review,
  ai_analyze_sentiment(review) AS sentiment
FROM (VALUES
  ('This product is absolutely amazing! Best purchase ever.'),
  ('Terrible experience. The item arrived broken and support was unhelpful.'),
  ('It works fine. Nothing special but gets the job done.'),
  ('The quality is good but shipping took way too long.')
) AS t(review);

## `ai_similarity()` — Compute Semantic Similarity

Computes a semantic similarity score (0 to 1) between two texts. Useful for deduplication, search relevance, and matching related content.

In [0]:
%sql
SELECT 
  text1, text2,
  ai_similarity(text1, text2) AS similarity_score
FROM (VALUES
  ('The cat sat on the mat', 'A feline was resting on a rug'),
  ('The cat sat on the mat', 'Stock prices rose 5% today'),
  ('Machine learning models', 'AI and deep learning algorithms'),
  ('I love pizza', 'I enjoy eating Italian flatbread with toppings')
) AS t(text1, text2);

## `ai_translate()` — Translate Text Between Languages

Translates text into a target language specified by a language code. Supports multilingual translation in a single query.

In [0]:
%sql
SELECT 
  original_text,
  ai_translate(original_text, 'it') AS italian,
  ai_translate(original_text, 'es') AS spanish,
  ai_translate(original_text, 'de') AS german,
  ai_translate(original_text, 'hu') AS hungarian,
  ai_translate(original_text, 'fr') AS french
FROM (VALUES
  ('Welcome to Databricks! Let us help you unlock the value of your data.')
) AS t(original_text);

## `ai_fix_grammar()`

Correct grammatical errors in text using a state-of-the-art generative AI model.

In [0]:
%sql
SELECT ai_fix_grammar('He go to school every days.') as fixed_grammar

## `ai_summarize()` — Summarize Long Text

Condenses long passages into concise summaries, preserving key information. Ideal for processing articles, documents, and verbose log entries.

In [0]:
%sql
SELECT ai_summarize(
  'Apache Spark is a unified analytics engine for large-scale data processing. It provides high-level APIs in Java, Scala, Python and R, and an optimized engine that supports general execution graphs. It also supports a rich set of higher-level tools including Spark SQL for SQL and structured data processing, pandas API on Spark for pandas workloads, MLlib for machine learning, GraphX for graph processing, and Structured Streaming for incremental computation and stream processing. Spark runs on Hadoop, Apache Mesos, Kubernetes, standalone, or in the cloud. It can access diverse data sources.'
) AS summary;

## `ai_mask()` — Mask Sensitive / PII Information

Masks personally identifiable information (PII) such as names, emails, and phone numbers. Useful for data anonymization before sharing or logging.

In [0]:
%sql
SELECT 
  original_text,
  ai_mask(original_text, array('person', 'email', 'phone')) AS masked_text
FROM (VALUES
  ('Please contact John Smith at john.smith@example.com or call 555-123-4567 for details.')
) AS t(original_text);

## Combining AI Functions — Chaining for Richer Analysis

AI functions compose naturally in a single query. Here we classify, extract, and analyze sentiment on customer messages simultaneously.

In [0]:
%sql
SELECT 
  message,
  ai_analyze_sentiment(message) AS sentiment,
  ai_classify(
    message,
    '["billing", "technical", "general"]',
    MAP('version', '2.0')
  ) AS category,
  ai_extract(message, '["product_name", "issue_description"]') AS extracted_info
FROM (VALUES
  ('My DataBot Pro subscription was charged twice this month and I want a refund immediately!'),
  ('The API connection keeps timing out when I try to upload files larger than 100MB'),
  ('When will you release the new dashboard feature you announced last quarter?')
) AS t(message);